---

## **Work**

**Step 1.1 - Step 1.6:**
* Environment setup -> Load PDF documents -> Extract text by heading/section structure (using helper function) -> Chunk documents into sections

**Step 1.7 - Step 2.2**
* Process parsed PDF content -> Enrich metadata (`source`, `section`, `subsection`, `document_type`) -> Generate embeddings -> Store chunks in Chroma vector database -> Implement query transformation

**Step 2.3 - Step 3.0**
* Topic/query type detection -> Metadata filtering -> Retrieve relevant chunks -> Rule-based filtering and deduplication -> Context compression -> Grounded answer generation

---

## **RAG Method**

This project implements a **modular function-based linear RAG workflow** for an Employment Compliance RAG Assistant that answers questions based on a Malaysian company handbook and the Employment Act 1955. The pipeline first transforms the user query, then retrieves relevant chunks from both the company handbook and the Employment Act using vector similarity search with topic metadata filtering. To improve recall for domain-specific legal questions, the system also applies rule-based retrieval augmentation through handcrafted query-to-section matching. Retrieved chunks are then deduplicated, filtered, and compressed before being passed into grounded answer generation. The pipeline supports single-source answering, dual-source comparison, and out-of-scope rejection, with final answers returned together with structured source references.


## **Step 1: Ingestion**

This step prepares the source documents for retrieval.
The PDFs are loaded, their text is extracted, headings and subheadings are identified, and the content is split into metadata-rich chunks before being stored in Chroma for semantic search.

#### **1.1 Environment setup**
This subsection installs the required Python packages for the RAG pipeline, including LangChain, Chroma, PDF processing tools, and OpenAI integration.

In [ ]:
# Colab setup
!pip -q install -U \
    langchain \
    langchain-openai \
    langchain-community \
    langchain-text-splitters \
    langchain-chroma \
    chromadb \
    pdfplumber \
    pypdf \
    pandas \
    tiktoken

#### **1.2 Imports and configuration**
This subsection imports the libraries used in the notebook and sets up the required configuration, such as API key access and reusable constants.

In [ ]:
import os
import re
import json
import shutil
import sys
import ipywidgets as widgets

import pandas as pd
import pdfplumber

from getpass import getpass
try:
    from google.colab import files
except ImportError:
    files = None
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain_community.callbacks import get_openai_callback
from IPython.display import display, Markdown

#### **1.3 Load the source PDFs and set API key**
This subsection loads the selected PDF documents, namely a Malaysian company handbook and the Employment Act 1955, which will serve as the knowledge base for the assistant.

Use these canonical file paths when available:
- `data/handbook.pdf`
- `data/employment_act.pdf`

If those files are not present, the notebook automatically detects uploaded or local PDFs using generic filename patterns.

This notebook uses:
- **OpenAI embeddings** for vector indexing
- **Chroma** for local persistent storage



In [ ]:
# Load the selected source PDFs
HANDBOOK_PATH = "data/handbook.pdf"
ACT_PATH = "data/employment_act.pdf"

PDF_SEARCH_DIRS = ["data", "."]


def normalize_filename(name: str) -> str:
    return re.sub(r"[^a-z0-9]+", " ", os.path.basename(name).lower()).strip()


def find_pdf_by_patterns(preferred_path: str, patterns, uploaded_files=None):
    candidates = []

    if os.path.exists(preferred_path):
        return preferred_path

    if uploaded_files:
        candidates.extend(name for name in uploaded_files if name.lower().endswith(".pdf"))

    for folder in PDF_SEARCH_DIRS:
        if os.path.isdir(folder):
            for name in os.listdir(folder):
                full_path = os.path.join(folder, name)
                if os.path.isfile(full_path) and name.lower().endswith(".pdf"):
                    candidates.append(full_path)

    unique_candidates = list(dict.fromkeys(candidates))

    for candidate in unique_candidates:
        normalized = normalize_filename(candidate)
        if any(pattern in normalized for pattern in patterns):
            return candidate

    return None


HANDBOOK_PATH = find_pdf_by_patterns(
    HANDBOOK_PATH,
    patterns=["employee handbook", "handbook", "hanbook"]
)
ACT_PATH = find_pdf_by_patterns(
    ACT_PATH,
    patterns=["employment act", "akta kerja", "akta 265"]
)

if (not HANDBOOK_PATH or not ACT_PATH) and files is not None:
    uploaded = files.upload()
    HANDBOOK_PATH = HANDBOOK_PATH or find_pdf_by_patterns(
        "data/handbook.pdf",
        patterns=["employee handbook", "handbook", "hanbook"],
        uploaded_files=uploaded.keys()
    )
    ACT_PATH = ACT_PATH or find_pdf_by_patterns(
        "data/employment_act.pdf",
        patterns=["employment act", "akta kerja", "akta 265"],
        uploaded_files=uploaded.keys()
    )

if not HANDBOOK_PATH:
    raise ValueError(
        "Company handbook PDF not found. Use data/handbook.pdf or a PDF filename containing 'handbook' or 'employee handbook'."
    )

if not ACT_PATH:
    raise ValueError(
        "Employment Act PDF not found. Use data/employment_act.pdf or a PDF filename containing 'employment act', 'akta kerja', or 'akta 265'."
    )

print("Using handbook PDF:", HANDBOOK_PATH)
print("Using Employment Act PDF:", ACT_PATH)

# Set OpenAI API key
if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("Enter OPENAI_API_KEY: ")

print("API key is set.")


#### **1.4 Helper functions for legal-document ingestion**
This subsection defines reusable helper functions for text normalization, page extraction, and metadata sanitization that are shared across both source documents.

Processes covered:
*   Cleans extracted text
*   Extracts page-by-page text
*   Tracks page ranges
*   Removes temporary page markers
*   Sanitized metadata for Chroma/LangChain




In [ ]:
def normalize_text(text: str) -> str:
    text = (text or "").replace("\xa0", " ")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def extract_pdf_pages(pdf_path: str):
    pages = []
    with pdfplumber.open(pdf_path) as pdf:
        for i, page in enumerate(pdf.pages, start=1):
            text = page.extract_text(x_tolerance=1, y_tolerance=3) or ""
            pages.append({
                "page": i,
                "text": normalize_text(text)
            })
    return pages


def page_span_from_text(text: str):
    pages = [int(x) for x in re.findall(r"\[\[PAGE=(\d+)\]\]", text)]
    if not pages:
        return None, None, ""
    return min(pages), max(pages), ",".join(str(p) for p in sorted(set(pages)))


def remove_page_markers(text: str) -> str:
    text = re.sub(r"\[\[PAGE=\d+\]\]\n?", "", text)
    return normalize_text(text)


def sanitize_metadata(meta: dict):
    clean = {}
    for key, value in meta.items():
        if value is None:
            clean[key] = ""
        elif isinstance(value, (str, int, float, bool)):
            clean[key] = value
        else:
            clean[key] = json.dumps(value, ensure_ascii=False)
    return clean

#### **1.5 Employment Act 1955 parsing helpers**
This subsection defines Act-specific helper functions to detect the legal table of contents, parse Parts and sections, separate the main body from schedules, and construct section-level blocks with enriched metadata.

Processes covered:
*   Cleans noisy TOC lines
*   Detects Part headings
*   Parses section numbers and titles
*   Locates the main Act body
*   Separate schedules
*   Create section-level blocks
*   Attaches structured legal metadata

In [ ]:
def clean_toc_line(line: str) -> str:
    line = line.strip()
    if not line:
        return ""
    # Remove obvious page headers/footers/noise
    if re.fullmatch(r"\d+", line):
        return ""
    if re.fullmatch(r"Employment \d+", line):
        return ""
    if re.fullmatch(r"Laws of Malaysia ACT 265", line):
        return ""
    if re.fullmatch(r"\d+ Laws of Malaysia ACT 265", line):
        return ""
    if line == "\\":
        return ""
    return line


PART_RE = re.compile(r"^PART\s+([IVXLCDM]+[A-Z]?)$", re.IGNORECASE)
SECTION_TOC_RE = re.compile(r"^(\d+[A-Z]?(?:-\d+[A-Z]?)?)\.\s+(.+)$")


def detect_toc_pages(pages):
    toc_pages = []
    in_toc = False
    for item in pages:
        text = item["text"]
        if "ARRANGEMENT OF SECTIONS" in text:
            in_toc = True
        if in_toc:
            toc_pages.append(item)
        if in_toc and "SECOND SCHEDULE" in text:
            break
    return toc_pages


def parse_toc_structure(toc_pages):
    raw_lines = []
    for page in toc_pages:
        raw_lines.extend(page["text"].splitlines())

    lines = [clean_toc_line(line) for line in raw_lines]
    lines = [line for line in lines if line]

    parts = []
    sections = []
    current_part = {"part_id": "", "part_title": ""}

    i = 0
    while i < len(lines):
        line = lines[i]

        part_match = PART_RE.match(line)
        section_match = SECTION_TOC_RE.match(line)

        if part_match:
            part_id = part_match.group(1).upper()
            title_lines = []
            j = i + 1
            while j < len(lines):
                nxt = lines[j]
                if PART_RE.match(nxt) or SECTION_TOC_RE.match(nxt):
                    break
                if nxt.lower() == "section":
                    j += 1
                    continue
                title_lines.append(nxt)
                j += 1

            current_part = {
                "part_id": part_id,
                "part_title": " ".join(title_lines).strip()
            }
            parts.append(current_part.copy())
            i = j
            continue

        if section_match:
            section_number = section_match.group(1)
            section_title = section_match.group(2).strip()

            continuation = []
            j = i + 1
            while j < len(lines):
                nxt = lines[j]
                if PART_RE.match(nxt) or SECTION_TOC_RE.match(nxt):
                    break
                if nxt.lower() == "section":
                    j += 1
                    continue
                continuation.append(nxt)
                j += 1

            full_title = " ".join([section_title] + continuation).strip()

            sections.append({
                "part_id": current_part.get("part_id", ""),
                "part_title": current_part.get("part_title", ""),
                "section_number": section_number,
                "section_title": full_title
            })
            i = j
            continue

        i += 1

    # Deduplicate repeated lines if any
    deduped_sections = []
    seen = set()
    for row in sections:
        key = (row["section_number"], row["section_title"])
        if key not in seen:
            seen.add(key)
            deduped_sections.append(row)

    section_map = {
        row["section_number"]: row
        for row in deduped_sections
    }

    return parts, deduped_sections, section_map


def find_body_start_page(pages):
    for item in pages:
        if "An Act relating to employment." in item["text"]:
            return item["page"]
    return 13  # fallback for this source document


def build_text_with_page_markers(pages, start_page):
    chunks = []
    for item in pages:
        if item["page"] >= start_page:
            chunks.append(f"[[PAGE={item['page']}]]\n{item['text']}")
    return "\n\n".join(chunks)


def split_main_text_and_schedules(body_text: str):
    match = re.search(r"(?m)^\s*FIRST SCHEDULE\s*$", body_text)
    if match:
        return body_text[:match.start()].strip(), body_text[match.start():].strip()
    return body_text.strip(), ""


SECTION_START_RE = re.compile(r"(?m)^(?P<section_number>\d+[A-Z]?)\.\s")


def infer_act_topic(part_title: str, section_title: str, text: str) -> str:
    s = f"{part_title} {section_title}".lower()
    t = text.lower()

    # highest priority: specific attendance / absenteeism / hours queries
    if any(x in s or x in t for x in [
        "absent", "absence", "attendance",
        "two consecutive working days",
        "hours of work", "working at night", "rest day"
    ]):
        return "attendance"

    if any(x in s or x in t for x in [
        "annual leave", "sick leave", "maternity", "paternity",
        "holiday", "public holiday"
    ]):
        return "leave"

    if any(x in s or x in t for x in [
        "lawful deductions", "payment of wages", "wages",
        "overtime", "hourly rate of pay"
    ]):
        return "remuneration"

    if any(x in s or x in t for x in [
        "termination", "notice of termination",
        "contract of service", "retirement",
        "lay-off", "redundancy"
    ]):
        return "termination"

    if any(x in s or x in t for x in [
        "flexible working arrangement"
    ]):
        return "employment_policy"

    return "general"


def extract_section_blocks(main_text: str, section_map: dict):
    blocks = []
    matches = list(SECTION_START_RE.finditer(main_text))

    for idx, match in enumerate(matches):
        start = match.start()
        end = matches[idx + 1].start() if idx + 1 < len(matches) else len(main_text)

        raw_block = main_text[start:end].strip()
        section_number = match.group("section_number")
        first_page, last_page, page_span = page_span_from_text(raw_block)
        clean_block = remove_page_markers(raw_block)

        mapped = section_map.get(section_number, {})

        metadata = {
            "document_title": "Employment Act 1955",
            "act_number": "Act 265",
            "source_file": ACT_PATH,
            "effective_as_of": "2023-01-01",
            "jurisdiction": "Malaysia",
            "part_id": mapped.get("part_id", ""),
            "part_title": mapped.get("part_title", ""),
            "section_number": section_number,
            "section_title": mapped.get("section_title", ""),
            "heading_path": (
                f"{mapped.get('part_id', '')} {mapped.get('part_title', '')} > "
                f"{section_number}. {mapped.get('section_title', '')}"
            ).strip(),
            "first_page": first_page if first_page is not None else -1,
            "last_page": last_page if last_page is not None else -1,
            "page_span": page_span,
            "block_type": "section",
            "granularity": "section",
            "is_deleted_or_omitted": bool(
                re.search(
                    r"\bdeleted\b|\bomitted\b",
                    mapped.get("section_title", ""),
                    re.IGNORECASE
                )
            ),
            "document_type": "law",
            "topic": infer_act_topic(
                mapped.get("part_title", ""),
                mapped.get("section_title", ""),
                clean_block
            ),
        }

        blocks.append({
            "text": clean_block,
            "metadata": metadata
        })

    return blocks


def extract_front_matter_blocks(pages, body_start_page):
    front_pages = [p for p in pages if p["page"] < body_start_page]
    blocks = []

    # Pages 1-2 contain title/version/applicability information
    for item in front_pages[:2]:
        if item["text"].strip():
            metadata = {
                "document_title": "Employment Act 1955",
                "act_number": "Act 265",
                "source_file": ACT_PATH,
                "effective_as_of": "2023-01-01",
                "jurisdiction": "Malaysia",
                "part_id": "FRONT",
                "part_title": "Front Matter",
                "section_number": "",
                "section_title": f"Front Matter Page {item['page']}",
                "heading_path": f"Front Matter > Page {item['page']}",
                "first_page": item["page"],
                "last_page": item["page"],
                "page_span": str(item["page"]),
                "block_type": "front_matter",
                "granularity": "section",
                "is_deleted_or_omitted": False,
                "document_type": "law",
                "topic": "general",
            }
            blocks.append({
                "text": item["text"],
                "metadata": metadata
            })

    return blocks


def extract_schedule_blocks(schedule_text: str):
    if not schedule_text.strip():
        return []

    split_re = re.compile(r"(?m)^(FIRST SCHEDULE|SECOND SCHEDULE)\s*$")
    matches = list(split_re.finditer(schedule_text))
    blocks = []

    if not matches:
        first_page, last_page, page_span = page_span_from_text(schedule_text)
        blocks.append({
            "text": remove_page_markers(schedule_text),
            "metadata": {
                "document_title": "Employment Act 1955",
                "act_number": "Act 265",
                "source_file": ACT_PATH,
                "effective_as_of": "2023-01-01",
                "jurisdiction": "Malaysia",
                "part_id": "SCHEDULE",
                "part_title": "Schedules",
                "section_number": "",
                "section_title": "Schedules",
                "heading_path": "Schedules",
                "first_page": first_page if first_page is not None else -1,
                "last_page": last_page if last_page is not None else -1,
                "page_span": page_span,
                "block_type": "schedule",
                "granularity": "section",
                "is_deleted_or_omitted": False,
                "document_type": "law",
                "topic": "general",
            }
        })
        return blocks

    for idx, match in enumerate(matches):
        start = match.start()
        end = matches[idx + 1].start() if idx + 1 < len(matches) else len(schedule_text)

        label = match.group(1).strip()
        raw_block = schedule_text[start:end].strip()
        first_page, last_page, page_span = page_span_from_text(raw_block)

        blocks.append({
            "text": remove_page_markers(raw_block),
            "metadata": {
                "document_title": "Employment Act 1955",
                "act_number": "Act 265",
                "source_file": ACT_PATH,
                "effective_as_of": "2023-01-01",
                "jurisdiction": "Malaysia",
                "part_id": "SCHEDULE",
                "part_title": "Schedules",
                "section_number": "",
                "section_title": label.title(),
                "heading_path": f"Schedules > {label.title()}",
                "first_page": first_page if first_page is not None else -1,
                "last_page": last_page if last_page is not None else -1,
                "page_span": page_span,
                "block_type": "schedule",
                "granularity": "section",
                "is_deleted_or_omitted": False,
                "document_type": "law",
                "topic": "general",
            }
        })

    return blocks

#### **1.6 Chunking and document-building helpers**
This subsection converts parsed text blocks into LangChain `Document` objects and splits longer sections into smaller metadata-rich chunks for retrieval.

Processes covered:
*   Splits long text blocks into smaller chunks
*   Preserves metadata during chunking
*   Creates unique chunk identifiers
*   Records chunk-level character counts
*   Converts parsed blocks into LangChain `Document` objects
*   Prepares chunked and section-level documents for indexing



In [ ]:
def build_chunk_documents(section_level_blocks, chunk_size=1200, chunk_overlap=200):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ". ", "; ", " "]
    )

    chunk_docs = []

    for block in section_level_blocks:
        base_meta = block["metadata"].copy()
        base_meta["granularity"] = "chunk"

        chunks = splitter.split_text(block["text"])
        for idx, chunk in enumerate(chunks):
            meta = base_meta.copy()
            meta["chunk_index"] = idx
            meta["chunk_id"] = (
                f"{meta.get('part_id', 'NA')}_"
                f"{meta.get('section_number', 'X')}_"
                f"{idx}_"
                f"{meta.get('first_page', -1)}"
            )
            meta["char_count"] = len(chunk)
            meta = sanitize_metadata(meta)

            chunk_docs.append(
                Document(
                    page_content=chunk,
                    metadata=meta
                )
            )

    return chunk_docs


def build_section_documents(section_level_blocks):
    section_docs = []
    for block in section_level_blocks:
        meta = sanitize_metadata(block["metadata"])
        section_docs.append(
            Document(
                page_content=block["text"],
                metadata=meta
            )
        )
    return section_docs

#### **1.7 Parse the company handbook and inspect the extracted structure**
This subsection extracts page text from the company handbook, detects the handbook's section and subsection hierarchy, and constructs metadata-rich handbook blocks for later chunking and retrieval. The handbook should contain clearly numbered policy sections such as employment policy, working hours and attendance, remuneration and allowance, and leave and public holidays.

Processes covered:
*   Extracts page text from the company handbook
*   Detects main section headings
*   Detects subsection and sub-subsection headings
*   Preserves heading context across pages
*   Creates handbook text blocks with metadata
*   Inspects the extracted handbook structure



In [ ]:
HANDBOOK_SECTION_RE = re.compile(r"(?m)^SECTION\s+(\d+)\s+[–-]\s+(.+)$")
HANDBOOK_SUBSECTION_RE = re.compile(r"(?m)^(\d+\.\d+(?:\.\d+)?)\s+(.+)$")


def find_handbook_body_start_page(pages):
    for item in pages:
        text = item["text"]

        # Skip table of contents pages
        if "TABLE OF CONTENTS" in text:
            continue

        # Look for the real body start
        if "SECTION 1" in text and "INTRODUCTION & INTERPRETATION" in text:
            return item["page"]

    return 7


def infer_handbook_topic(section: str, subsection: str, text: str) -> str:
    s = f"{section} {subsection}".lower()
    t = text.lower()

    if subsection.startswith(("5.", "6.4")) or any(x in s for x in ["leave", "maternity", "paternity", "holiday"]):
        return "leave"

    if subsection.startswith(("4.", "7.", "8.1.4")) or any(x in s for x in ["remuneration", "allowance", "deduction", "wages", "salary", "overtime"]):
        return "remuneration"

    if subsection.startswith(("3.",)) or any(x in s for x in ["working hours", "attendance", "punctuality", "absenteeism"]):
        return "attendance"

    if subsection.startswith(("2.11", "2.12", "2.13", "2.14")) or any(x in s for x in ["termination", "retirement", "retrenchment", "separation"]):
        return "termination"

    if subsection.startswith(("2.1", "2.2", "2.3", "2.4", "2.5", "2.6", "2.7", "2.8", "2.9", "2.10")):
        return "employment_policy"

    if any(x in t for x in ["maternity", "paternity", "medical leave", "annual leave", "sick leave"]):
        return "leave"
    if any(x in t for x in ["salary", "wage", "allowance", "deduction", "overtime", "epf"]):
        return "remuneration"
    if any(x in t for x in ["working hours", "attendance", "punctuality", "absenteeism", "absence"]):
        return "attendance"
    if any(x in t for x in ["termination", "retirement", "retrenchment", "resignation", "notice period"]):
        return "termination"

    return "general"


def extract_handbook_blocks(pages):
    blocks = []
    current_section = ""
    body_start_page = find_handbook_body_start_page(pages)

    for item in pages:
        page_num = item["page"]
        text = item["text"]

        # skip cover / TOC / front matter
        if page_num < body_start_page:
            continue

        if not text.strip():
            continue

        section_matches = list(HANDBOOK_SECTION_RE.finditer(text))
        if section_matches:
            sec_num = section_matches[0].group(1).strip()
            sec_title = section_matches[0].group(2).strip()
            current_section = f"{sec_num}.0 {sec_title}"

        subsection_matches = list(HANDBOOK_SUBSECTION_RE.finditer(text))

        if not subsection_matches:
            metadata = {
                "document_title": "Company Handbook",
                "source_file": HANDBOOK_PATH,
                "document_type": "handbook",
                "section": current_section,
                "section_number": current_section.split()[0] if current_section else "",
                "subsection": "",
                "subsection_number": "",
                "page": page_num,
                "topic": infer_handbook_topic(current_section, "", text),
                "block_type": "page_block",
                "granularity": "section"
            }
            blocks.append({"text": text, "metadata": metadata})
            continue

        for idx, match in enumerate(subsection_matches):
            start = match.start()
            end = subsection_matches[idx + 1].start() if idx + 1 < len(subsection_matches) else len(text)

            raw_block = text[start:end].strip()
            if not raw_block:
                continue

            sub_num = match.group(1).strip()
            sub_title = match.group(2).strip()
            subsection_label = f"{sub_num} {sub_title}"

            metadata = {
                "document_title": "Company Handbook",
                "source_file": HANDBOOK_PATH,
                "document_type": "handbook",
                "section": current_section,
                "section_number": current_section.split()[0] if current_section else "",
                "subsection": subsection_label,
                "subsection_number": sub_num,
                "page": page_num,
                "topic": infer_handbook_topic(current_section, subsection_label, raw_block),
                "block_type": "subsection_block",
                "granularity": "section"
            }

            blocks.append({"text": raw_block, "metadata": metadata})

    return blocks


handbook_pages = extract_pdf_pages(HANDBOOK_PATH)
handbook_blocks = extract_handbook_blocks(handbook_pages)

#### **1.8 Parse the Employment Act 1955 and inspect the extracted structure**
This subsection extracts page text from the Employment Act 1955, detects the Arrangement of Sections, and parses the legal hierarchy into Parts, sections, and metadata mappings for later chunk construction. This is important because the Act is organized by formal legal structure rather than handbook-style policy headings.

Processes covered:
*   Extracts page text from the Employment Act 1955
*   Detects the *Arrangement of Sections*
*   Finds the start of the main Act body
*   Parses the hierarchy into Parts and sections
*   Builds a section metadata mapping for later retrieval



In [ ]:
act_pages = extract_pdf_pages(ACT_PATH)
act_body_start_page = find_body_start_page(act_pages)
act_toc_pages = detect_toc_pages(act_pages)

act_parts, act_toc_sections, act_section_map = parse_toc_structure(act_toc_pages)

print("Total Act pages:", len(act_pages))
print("Detected Act body start page:", act_body_start_page)
print("Detected Act TOC pages:", [p["page"] for p in act_toc_pages])
print("Parsed Act parts:", len(act_parts))
print("Parsed Act TOC sections:", len(act_toc_sections))

act_parts_df = pd.DataFrame(act_parts)
act_sections_df = pd.DataFrame(act_toc_sections)

display(act_parts_df)
display(act_sections_df.head(20))

#### **1.9 Create metadata-rich chunks**
This subsection converts the parsed handbook and Employment Act 1955 structures into metadata-rich text blocks and then splits them into chunked LangChain `Document` objects. These chunked documents will form the final retrieval corpus for the Chroma vector store.

Processes covered:
*   Builds section-level blocks for the Employment Act 1955
*   Reuses handbook blocks extracted earlier
*   Converts parsed blocks into section-level documents
*   Creates chunked documents for both sources
*   Merges all chunked documents into one retrieval corpus

In [ ]:
# Build Employment Act section-level blocks
act_body_text = build_text_with_page_markers(act_pages, act_body_start_page)
act_main_text, act_schedule_text = split_main_text_and_schedules(act_body_text)

act_front_blocks = extract_front_matter_blocks(act_pages, act_body_start_page)
act_section_blocks = extract_section_blocks(act_main_text, act_section_map)
act_schedule_blocks = extract_schedule_blocks(act_schedule_text)

act_all_blocks = act_front_blocks + act_section_blocks + act_schedule_blocks

print("Act front blocks:", len(act_front_blocks))
print("Act section blocks:", len(act_section_blocks))
print("Act schedule blocks:", len(act_schedule_blocks))
print("Total Act blocks:", len(act_all_blocks))


# Convert handbook blocks into LangChain Documents
handbook_section_docs = build_section_documents(handbook_blocks)
handbook_chunk_docs = build_chunk_documents(
    handbook_blocks,
    chunk_size=1000,
    chunk_overlap=150
)

print("Handbook section-level docs:", len(handbook_section_docs))
print("Handbook chunked docs:", len(handbook_chunk_docs))


# Convert Employment Act blocks into LangChain Documents
act_section_docs = build_section_documents(act_all_blocks)
act_chunk_docs = build_chunk_documents(
    act_all_blocks,
    chunk_size=1200,
    chunk_overlap=200
)

print("Act section-level docs:", len(act_section_docs))
print("Act chunked docs:", len(act_chunk_docs))

all_chunk_docs = handbook_chunk_docs + act_chunk_docs
all_section_docs = handbook_section_docs + act_section_docs

print("Total section-level docs:", len(all_section_docs))
print("Total chunked docs for retrieval:", len(all_chunk_docs))


chunk_preview = pd.DataFrame([
    {
        "source_file": doc.metadata.get("source_file", ""),
        "document_type": doc.metadata.get("document_type", ""),
        "section": doc.metadata.get("section", doc.metadata.get("part_title", "")),
        "subsection": doc.metadata.get("subsection", doc.metadata.get("section_title", "")),
        "page": doc.metadata.get("page", doc.metadata.get("first_page", "")),
        "topic": doc.metadata.get("topic", ""),
        "chunk_id": doc.metadata.get("chunk_id", ""),
        "char_count": doc.metadata.get("char_count", "")
    }
    for doc in all_chunk_docs[:20]
])

display(chunk_preview)

#### **1.10 Build the Chroma vector store**
This subsection generates embeddings for all chunked documents and stores them in a local Chroma vector database. The vector store will later be used for similarity-based retrieval in the RAG workflow.

Processes covered:
*   Initializes the embedding model
*   Resets the local Chroma persistence directory
*   Stores all chunked documents in one collection
*   Prepares the vector store for later retrieval



In [ ]:
import gc
import os
import shutil
import uuid

for var_name in [
    "handbook_vectorstore", "act_vectorstore",
    "handbook_retriever", "act_retriever", "embeddings"
]:
    if var_name in globals():
        del globals()[var_name]

gc.collect()

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

handbook_dir = f"/tmp/chroma_handbook_{uuid.uuid4().hex}"
act_dir = f"/tmp/chroma_act_{uuid.uuid4().hex}"

for folder in [handbook_dir, act_dir]:
    if os.path.exists(folder):
        shutil.rmtree(folder, ignore_errors=True)

handbook_vectorstore = Chroma.from_documents(
    documents=handbook_chunk_docs,
    embedding=embeddings,
    persist_directory=handbook_dir,
    collection_name=f"handbook_collection_{uuid.uuid4().hex[:8]}"
)

act_vectorstore = Chroma.from_documents(
    documents=act_chunk_docs,
    embedding=embeddings,
    persist_directory=act_dir,
    collection_name=f"act_collection_{uuid.uuid4().hex[:8]}"
)

handbook_retriever = handbook_vectorstore.as_retriever(search_kwargs={"k": 6})
act_retriever = act_vectorstore.as_retriever(search_kwargs={"k": 6})

print("Handbook vector store ready:", len(handbook_chunk_docs))
print("Act vector store ready:", len(act_chunk_docs))
print("Handbook dir:", handbook_dir)
print("Act dir:", act_dir)

## **Step 2: RAG Workflow**
This step implements the retrieval-based question answering pipeline for the assistant. A user query is first reformulated to improve retrieval quality, then relevant chunks are retrieved from the Chroma vector store using metadata-aware filtering. The retrieved context is compressed to reduce noise, and the final answer is generated by the language model based only on the supporting evidence from the company handbook and the Employment Act 1955.


#### **2.1 Initialize the LLM**
This subsection initializes the language model used in the RAG workflow. The model is responsible for two main tasks: rewriting user questions into retrieval-friendly queries and generating grounded answers based only on the retrieved context from the Employee Handbook and the Employment Act.

Processes covered:
*   Initializes the chat model
*   Sets a low temperature for stable and consistent outputs



In [ ]:
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

#### **2.2 Query transformation**
This subsection reformulates the user’s original question into a shorter and more retrieval-focused query. The purpose of this step is to improve retrieval quality by emphasizing the key employment terms, policy concepts, and legal keywords before searching the Chroma vector store.

Processes covered:
*   Accepts the original user query
*   Rewrites the query into a retrieval-friendly form
*   Preserves the key employment and policy terms
*   Prepares the transformed query for semantic retrieval

In [ ]:
chat_history = []


def reset_chat_history():
    """Clear conversation memory for a new QA session."""
    chat_history.clear()


def format_chat_history_for_prompt(history, max_turns: int = 5) -> str:
    recent_turns = history[-max_turns:]
    if not recent_turns:
        return "No previous conversation."

    lines = []
    for turn in recent_turns:
        user = str(turn.get("user", "")).strip()
        assistant = str(turn.get("assistant", "")).strip()
        lines.append(f"User: {user}")
        lines.append(f"Assistant: {assistant}")

    return "\n".join(lines)


def is_true_follow_up_question(user_query: str) -> bool:
    q = user_query.strip().lower()
    if not q:
        return False

    follow_up_markers = [
        "what about", "how about", "and if", "what if", "does it", "do they",
        "is it", "are they", "can it", "can they", "that policy", "this policy",
        "the policy", "that leave", "this leave", "those rules", "these rules",
        "same", "above", "previous", "earlier", "there", "then"
    ]
    pronoun_starts = ("it ", "that ", "this ", "they ", "them ", "those ", "these ")

    return q.startswith(pronoun_starts) or any(marker in q for marker in follow_up_markers)


def rewrite_standalone_question(user_query: str, history=None) -> str:
    history = chat_history if history is None else history
    original_query = user_query.strip()

    if not history or not is_true_follow_up_question(original_query):
        return original_query

    history_text = format_chat_history_for_prompt(history)

    prompt = f"""
You rewrite only true follow-up questions for an HR RAG chatbot.

Task:
Use the chat history only to resolve explicit references in the latest user question.

Rules:
- Preserve the latest user's intent exactly.
- Resolve pronouns and references such as "it", "that", "this", "there", "the policy", or "in the handbook" only when chat history clearly identifies them.
- Do not add "Employment Act", "Act", "law", "legal", "company policy", or "handbook" unless the latest question says it or the chat history clearly requires that same source constraint.
- If the latest question is already understandable by itself, return it unchanged.
- Do not answer the question.
- Output only the standalone question.

Chat history:
{history_text}

Latest user question:
{user_query}
"""
    response = llm.invoke(prompt)
    standalone_question = response.content.strip()

    if not standalone_question:
        return original_query

    original_lower = original_query.lower()
    rewritten_lower = standalone_question.lower()
    protected_terms = ["employment act", "akta kerja", "company policy", "handbook"]
    for term in protected_terms:
        if term not in original_lower and term in rewritten_lower:
            history_requires_term = any(term in str(turn.get("user", "")).lower() for turn in history[-2:])
            if not history_requires_term:
                return original_query

    return standalone_question


def transform_query(user_query: str) -> str:
    prompt = f"""
You are a query rewriting assistant for retrieval.

Task:
Rewrite the user's question into a short keyword-focused retrieval query.

Rules:
- Keep the original meaning unchanged.
- Preserve important employment and legal keywords exactly when possible.
- Do NOT introduce new concepts that are not in the original query.
- Do NOT add "Employment Act", "Act", "law", "legal", "company policy", or "handbook" unless the user used that source constraint.
- Do NOT replace "leave" with "absence".
- Do NOT replace "absence" with "leave".
- Do NOT replace "termination" with "retirement" or "retrenchment".
- Remove conversational wording.
- Output only one short retrieval query.

User query:
{user_query}
"""
    response = llm.invoke(prompt)
    transformed = response.content.strip() or user_query.strip()

    original_lower = user_query.lower()
    transformed_lower = transformed.lower()
    protected_terms = ["employment act", "akta kerja", "company policy", "handbook"]
    if any(term not in original_lower and term in transformed_lower for term in protected_terms):
        return user_query.strip()

    return transformed


#### **2.3 Topic detection and metadata filtering**
This subsection identifies the likely topic of the transformed query and uses metadata to guide retrieval toward the most relevant document chunks. By narrowing the search space to a specific topic, the assistant can reduce retrieval noise and improve the relevance of the supporting context.

Processes covered:
*   Detects the likely topic of the user query
*   Maps the query to a predefined policy or legal topic
*   Uses topic metadata to support more focused retrieval
*   Prepares filtered search settings for the retriever

In [ ]:
def contains_any(text: str, phrases) -> bool:
    text = text.lower()
    return any(phrase in text for phrase in phrases)


def detect_topic(query: str) -> str:
    q = query.lower()

    topic_keywords = {
        "leave": [
            "maternity", "paternity", "annual leave", "medical leave", "sick leave",
            "hospitalisation leave", "hospitalization leave", "marriage leave",
            "calamity leave", "unpaid leave", "leave entitlement", "holiday",
            "public holiday", "maternity allowance", "confinement"
        ],
        "attendance": [
            "working hours", "office hours", "attendance", "punctuality", "absenteeism",
            "absence", "absent", "late", "clock in", "clock out", "flexible working time",
            "flexible working", "working time", "skip work", "skips work", "without notice"
        ],
        "termination": [
            "termination", "resignation", "resign", "retirement", "retrenchment",
            "redundancy", "separation", "notice period", "quit", "dismissal",
            "misconduct", "layoff", "laid off", "leave the company"
        ],
        "remuneration": [
            "salary", "salaries", "wages", "wage", "allowance", "bonus", "deduction",
            "overtime", "epf", "socso", "eis", "pay", "payment", "pay cut",
            "salary cut", "ordinary rate of pay", "wage period"
        ],
        "employment_policy": [
            "recruitment", "probation", "confirmation", "performance appraisal",
            "promotion", "trial period", "on trial", "appointment", "contract type",
            "types of employment"
        ],
    }

    # Specific legal/HR leave terms should win over broad words like "pay" or "termination".
    if contains_any(q, topic_keywords["leave"]):
        return "leave"

    if contains_any(q, topic_keywords["termination"]):
        return "termination"

    if contains_any(q, topic_keywords["attendance"]):
        return "attendance"

    if contains_any(q, topic_keywords["remuneration"]):
        return "remuneration"

    if contains_any(q, topic_keywords["employment_policy"]):
        return "employment_policy"

    return "general"


def detect_requested_sources(query: str) -> set:
    q = query.lower()
    requested = set()

    if contains_any(q, [
        "handbook", "employee handbook", "company handbook", "company policy",
        "according to company", "according to the company", "company rule",
        "policy manual", "under company policy"
    ]):
        requested.add("handbook")

    if contains_any(q, [
        "employment act", "akta kerja", "what does the act", "under the act",
        "section 60", "section 60fa", "60fa", "section 37", "section 19",
        "section 15", "section 24", "section 12", "section 60a", "section 60e"
    ]):
        requested.add("act")

    return requested


def is_comparison_intent(query: str) -> bool:
    q = query.lower()
    return contains_any(q, [
        "compare", "comparison", "comply", "compliance", "align", "aligned",
        "consistent", "consistency", "difference between", "different from", "vs",
        "versus", "how about", "what about", "same as"
    ])


def extract_comparison_concepts(query: str):
    q = query.lower()
    concept_groups = [
        ("maternity leave", ["maternity", "maternity leave", "maternity allowance"]),
        ("paternity leave", ["paternity", "paternity leave", "60fa", "60 fa"]),
        ("annual leave", ["annual leave"]),
        ("medical leave", ["medical leave", "sick leave", "hospitalisation leave", "hospitalization leave"]),
        ("absenteeism", ["absenteeism", "absence", "absent", "skip work", "skips work"]),
        ("punctuality", ["punctuality", "late", "clock in", "clock out"]),
        ("working hours", ["working hours", "office hours", "normal office hours"]),
        ("overtime", ["overtime"]),
        ("resignation", ["resignation", "resign", "quit"]),
        ("termination", ["termination", "dismissal", "terminate"]),
        ("wage payment", ["wage payment", "wage payment policy", "payment of wages", "when wages paid", "wages paid", "wage period"]),
        ("deductions", ["deduction", "deductions", "salary cut", "pay cut"]),
    ]

    concepts = []
    for label, aliases in concept_groups:
        if any(alias in q for alias in aliases):
            concepts.append(label)

    if "leave" in q and "company policy" in q and "employment act" in q:
        for label in ["annual leave", "medical leave", "maternity leave", "paternity leave"]:
            if label not in concepts:
                concepts.append(label)

    return concepts



REQUIRED_EVIDENCE_RULES = {
    "paternity leave": {
        "triggers": ["paternity", "paternity leave", "60fa", "60 fa"],
        "handbook": ["5.7 paternity leave"],
        "act": ["section 60fa", "60fa", "60 fa", "paternity leave"],
    },
    "maternity leave": {
        "triggers": ["maternity", "maternity leave", "maternity allowance"],
        "handbook": ["5.5 maternity leave"],
        "act": ["section 37", "maternity leave", "maternity allowance", "eligible period"],
    },
    "overtime": {
        "triggers": ["overtime"],
        "handbook": ["4.5 overtime"],
        "act": ["section 60a", "60a", "hours of work and working at night", "one and half times"],
    },
    "wage payment": {
        "triggers": ["wage payment", "payment of wages", "wages paid", "wages be paid", "must wages", "when must wages", "wage period", "salary paid", "payday"],
        "handbook": ["4.1 payment of wages"],
        "act": ["section 19", "time of payment of wages"],
    },
    "absenteeism": {
        "triggers": ["absenteeism", "absence", "absent", "miss work", "skip work", "skips work", "two consecutive working days"],
        "handbook": ["3.4 absenteeism"],
        "act": ["section 15", "when contract is deemed to be broken by employer and employee", "two consecutive working days"],
    },
    "annual leave": {
        "triggers": ["annual leave", "carry forward annual leave"],
        "handbook": ["5.1 annual leave"],
        "act": ["section 60e", "60e", "annual leave"],
    },
    "lawful deductions": {
        "triggers": ["lawful deduction", "lawful deductions", "deduction", "deductions", "salary cut", "pay cut", "wages deducted", "deducted"],
        "handbook": ["4.4 employee deductions"],
        "act": ["section 24", "lawful deductions", "deductions from wages"],
    },
    "notice period": {
        "triggers": ["notice period", "notice of termination", "resign", "resignation", "quit"],
        "handbook": ["2.12 termination of employment", "notice period"],
        "act": ["section 12", "notice of termination of contract"],
    },
}


def required_evidence_topics(query: str):
    q = query.lower()
    topics = []
    for topic, spec in REQUIRED_EVIDENCE_RULES.items():
        if any(trigger in q for trigger in spec["triggers"]):
            topics.append(topic)
    return topics


REQUIRED_EVIDENCE_SECTIONS = {
    "paternity leave": {"handbook": ["5.7"], "act": ["60FA"]},
    "maternity leave": {"handbook": ["5.5"], "act": ["37"]},
    "overtime": {"handbook": ["4.5"], "act": ["60A"]},
    "wage payment": {"handbook": ["4.1"], "act": ["19"]},
    "absenteeism": {"handbook": ["3.4"], "act": ["15"]},
    "annual leave": {"handbook": ["5.1"], "act": ["60E"]},
    "lawful deductions": {"handbook": ["4.4"], "act": ["24"]},
    "notice period": {"handbook": ["2.12"], "act": ["12"]},
}


def doc_matches_required_evidence(doc, topic: str, source_name: str) -> bool:
    expected = [str(x).lower() for x in REQUIRED_EVIDENCE_SECTIONS.get(topic, {}).get(source_name, [])]
    if not expected:
        return False

    if source_name == "act":
        section_number = str(doc.metadata.get("section_number", "")).lower().replace(" ", "")
        section_title = str(doc.metadata.get("section_title", "")).lower()
        text = str(doc.page_content).lower()
        if topic == "paternity leave":
            return ("60 fa" in section_title or "paternity leave" in section_title) and any(
                phrase in text for phrase in ["paternity leave", "married male employee", "seven consecutive days"]
            )
        return any(section_number == value.lower().replace(" ", "") for value in expected)

    section = str(doc.metadata.get("section", "")).lower()
    subsection = str(doc.metadata.get("subsection", "")).lower()
    label = f"{section} {subsection}"
    return any(label.startswith(value) or f" {value}" in label for value in expected)


def is_out_of_scope_query(query: str) -> bool:
    q = query.lower()

    off_topic_phrases = [
        "capital of", "world cup", "fifa", "weather", "stock market", "poem",
        "translate", "calculus", "ceo of apple", "bake", "chocolate cake",
        "recipe", "cook", "cooking", "ingredient", "ingredients"
    ]
    if contains_any(q, off_topic_phrases):
        return True

    employment_terms = [
        "employment", "employee", "employer", "company", "handbook", "policy",
        "employment act", "akta kerja", "statutory", "legal", "lawful", "wages",
        "salary", "pay", "deduction", "overtime", "leave", "maternity", "paternity",
        "probation", "confirmation", "resign", "resignation", "termination",
        "retirement", "retrenchment", "notice period", "working hours", "attendance",
        "absenteeism", "punctuality", "flexible working", "pregnant", "holiday",
        "rest day", "public holiday", "allowance", "benefit", "benefits"
    ]

    if detect_topic(q) != "general":
        return False

    if contains_any(q, employment_terms):
        return False

    return True


def get_subsection_hints(query: str):
    q = query.lower()
    hints = []

    # termination / resignation / notice
    if any(x in q for x in ["notice period", "quit", "resign", "resignation", "termination"]):
        hints.extend([
            "2.12 TERMINATION OF EMPLOYMENT",
            "Notice of termination of contract",
            "Termination of contract without notice",
            "Termination of contract for special reasons"
    ])

    # retirement
    if any(x in q for x in ["retirement", "retire"]):
        hints.extend([
            "2.13 RETIREMENT",
            "Termination, lay-off and retirement benefits"
        ])

    # retrenchment / redundancy
    if any(x in q for x in ["retrenchment", "redundancy", "business closure", "layoff", "laid off"]):
        hints.extend([
            "2.14 RETRENCHMENT DUE TO REDUNDANCY OR BUSINESS CLOSURE",
            "Termination, lay-off and retirement benefits"
    ])

    # termination without notice
    if "termination without notice" in q:
        hints.extend([
            "Termination of contract without notice"
        ])

    # maternity
    if "maternity" in q:
        hints.extend([
            "5.5 MATERNITY LEAVE",
            "6.4 MATERNITY BENEFITS",
            "Length of eligible period and entitlement to maternity allowance",
            "Restriction on termination of pregnant female employee"
        ])

    # paternity / Employment Act section 60FA
    if "paternity" in q or "60fa" in q or "60 fa" in q:
        hints.extend([
            "5.7 PATERNITY LEAVE",
            "Paternity leave",
            "60FA Paternity leave",
            "60 FA. Paternity leave",
            "Sick leave 60 FA. Paternity leave",
            "married male employee",
            "seven consecutive days",
            "ordinary rate of pay",
            "same employer for at least twelve months",
            "employed by the same employer at least twelve months",
            "notified his employer",
            "confinement"
        ])

    # annual leave
    if "annual leave" in q:
        hints.extend([
            "5.1 ANNUAL LEAVE",
            "Annual leave"
        ])

    # medical / sick leave
    if any(x in q for x in ["medical leave", "sick leave"]):
        hints.extend([
            "5.2 MEDICAL LEAVE",
            "Sick leave"
        ])

    # deductions / pay cut
    if any(x in q for x in ["deduct", "deduction", "salary cut", "pay cut", "owe money"]):
        hints.extend([
            "4.4 EMPLOYEE DEDUCTIONS",
            "4.1 PAYMENT OF WAGES",
            "Lawful deductions",
            "Payment of wages",
            "Payment on termination of contract in special circumstances and on breach of contract"
        ])

    # wage payment timing
    if any(x in q for x in ["wages payment", "wage payment", "wage payment policy", "when wages paid", "wages paid", "wage payment date", "time of payment of wages", "wage period"]):
        hints.extend([
            "4.1 PAYMENT OF WAGES",
            "Time of payment of wages",
            "not later than the seventh day after the last day of any wage period",
            "paid on the 24th of the month"
        ])

    # office / working hours
    if any(x in q for x in ["working hours", "official working hours", "office hours", "normal office hours"]):
        hints.extend([
            "3.1 WORKING HOURS",
            "Hours of work and working at night"
        ])

    # flexible working
    if any(x in q for x in ["flexible working time", "flexible working", "flexible working arrangement"]):
        hints.extend([
            "3.1 WORKING HOURS",
            "Flexible working arrangement",
            "Application for flexible working arrangement"
        ])

    # absenteeism / punctuality
    if any(x in q for x in ["absent", "absence", "absenteeism", "two consecutive working days"]):
        hints.extend([
            "3.4 ABSENTEEISM",
            "When contract is deemed to be broken by employer and employee",
            "contract is deemed to be broken",
            "continuously absent from work for more than two consecutive working days"
        ])

    if "punctuality" in q or "late" in q:
        hints.extend([
            "3.3 PUNCTUALITY",
            "3.1 WORKING HOURS"
        ])

    # pregnant employee termination
    if "pregnant employee" in q or "terminate pregnant employee" in q:
        hints.extend([
            "Restriction on termination of pregnant female employee"
        ])

    # comparison concepts: add hints for each side, not just the broad topic
    concepts = extract_comparison_concepts(q)
    if len(concepts) >= 2 or is_comparison_intent(q):
        for concept in concepts:
            if concept == "maternity leave":
                hints.extend(["5.5 MATERNITY LEAVE", "Length of eligible period and entitlement to maternity allowance"])
            elif concept == "paternity leave":
                hints.extend(["5.7 PATERNITY LEAVE", "60FA Paternity leave", "Sick leave 60 FA. Paternity leave"])
            elif concept == "annual leave":
                hints.extend(["5.1 ANNUAL LEAVE", "Annual leave"])
            elif concept == "medical leave":
                hints.extend(["5.2 MEDICAL LEAVE", "Sick leave"])
            elif concept == "absenteeism":
                hints.extend(["3.4 ABSENTEEISM", "When contract is deemed to be broken by employer and employee"])
            elif concept == "punctuality":
                hints.extend(["3.3 PUNCTUALITY", "3.1 WORKING HOURS"])
            elif concept == "working hours":
                hints.extend(["3.1 WORKING HOURS", "Hours of work and working at night"])
            elif concept == "overtime":
                hints.extend(["4.5 OVERTIME", "Hours of work and working at night"])
            elif concept == "resignation":
                hints.extend(["2.12 TERMINATION OF EMPLOYMENT", "Notice of termination of contract"])
            elif concept == "termination":
                hints.extend(["2.12 TERMINATION OF EMPLOYMENT", "Termination of contract without notice"])
            elif concept == "wage payment":
                hints.extend(["4.1 PAYMENT OF WAGES", "Time of payment of wages"])
            elif concept == "deductions":
                hints.extend(["4.4 EMPLOYEE DEDUCTIONS", "Lawful deductions"])

    # handbook omission / law still applies
    if any(x in q for x in [
        "if the handbook does not mention",
        "handbook omission",
        "not mentioned in handbook",
        "not mentioned in the handbook",
        "can the employment act still apply"
    ]):
        hints.extend([
            "1.1 INTRODUCTION",
            "2.1 TYPES OF EMPLOYMENT"
        ])

    # overtime
    if "overtime" in q:
        hints.extend([
            "4.5 OVERTIME",
            "normal working day 1.5 ordinary rate of pay",
            "rest day overtime",
            "public holiday overtime",
            "Hours of work and working at night"
        ])

    return list(dict.fromkeys(hints))


def get_retrieval_filter(query: str):
    topic = detect_topic(query)
    if topic and topic != "general":
        return topic, {"topic": topic}
    return topic, None


#### **2.4 Retrieval from Chroma**
This subsection retrieves the most relevant chunks from the Chroma vector store using the transformed query. When a topic is detected, metadata filtering is applied so that retrieval is restricted to chunks that belong to the matching topic.

Processes covered:
*   Accepts the transformed query
*   Applies topic-based metadata filtering when available
*   Retrieves relevant chunks from Chroma
*   Returns the retrieved chunks for downstream compression and answer generation

In [ ]:
def deduplicate_docs(docs):
    seen = set()
    unique_docs = []

    for doc in docs:
        key = (
            doc.metadata.get("source_file", ""),
            doc.metadata.get("section", doc.metadata.get("part_title", "")),
            doc.metadata.get("subsection", doc.metadata.get("section_title", "")),
            doc.metadata.get("page", doc.metadata.get("first_page", "")),
            doc.page_content[:120]
        )
        if key not in seen:
            seen.add(key)
            unique_docs.append(doc)

    return unique_docs


def remove_invalid_handbook_docs(docs):
    cleaned = []

    for doc in docs:
        doc_type = str(doc.metadata.get("document_type", ""))
        page = doc.metadata.get("page", doc.metadata.get("first_page", 0))

        try:
            page = int(page)
        except:
            page = 0

        if doc_type == "handbook" and page < 7:
            continue

        cleaned.append(doc)

    return cleaned


def doc_label(doc):
    section = str(doc.metadata.get("section", doc.metadata.get("part_title", ""))).lower()
    subsection = str(doc.metadata.get("subsection", doc.metadata.get("section_title", ""))).lower()
    text = str(doc.page_content).lower()
    return f"{section} || {subsection} || {text}"


def find_handbook_docs_by_rule(query: str, limit: int = 6):
    q = query.lower()
    matched = []

    def add_if_match(doc, *patterns):
        label = doc_label(doc)
        if any(pattern.lower() in label for pattern in patterns):
            matched.append(doc)

    for doc in handbook_chunk_docs:
        if "maternity" in q:
            add_if_match(doc, "5.5 maternity leave", "maternity leave", "6.4 maternity benefits")

        if "paternity" in q or "60fa" in q or "60 fa" in q:
            add_if_match(doc, "5.7 paternity leave", "paternity leave")

        if "annual leave" in q:
            add_if_match(doc, "5.1 annual leave", "annual leave")

        if any(x in q for x in ["medical leave", "sick leave", "hospitalisation leave", "hospitalization leave"]):
            add_if_match(doc, "5.2 medical leave", "medical leave", "sick leave")

        if any(x in q for x in ["working hours", "office hours", "normal office hours"]):
            add_if_match(doc, "3.1 working hours", "working hours")

        if "overtime" in q:
            add_if_match(doc, "4.5 overtime", "overtime")

        if "punctuality" in q or "late" in q:
            add_if_match(doc, "3.3 punctuality", "punctuality")

        if any(x in q for x in ["absenteeism", "absence", "absent", "skip work", "skips work"]):
            add_if_match(doc, "3.4 absenteeism", "absenteeism")

        if any(x in q for x in ["notice period", "quit", "resign", "resignation", "termination"]):
            add_if_match(doc, "2.12 termination of employment", "termination of employment", "notice period")

        if any(x in q for x in ["deduction", "deductions", "salary cut", "pay cut"]):
            add_if_match(doc, "4.4 employee deductions", "employee deductions")

        if any(x in q for x in ["wage payment", "wage payment policy", "payment of wages", "when wages paid", "wages paid", "wage period"]):
            add_if_match(doc, "4.1 payment of wages", "payment of wages")

    return deduplicate_docs(matched)[:limit]


def comparison_expansion_queries(query: str):
    concepts = extract_comparison_concepts(query)
    expansions = []

    mapping = {
        "maternity leave": "maternity leave maternity benefits eligible period maternity allowance",
        "paternity leave": "paternity leave section 60FA seven consecutive days ordinary rate of pay married male employee same employer twelve months notified employer confinement",
        "annual leave": "annual leave entitlement",
        "medical leave": "medical leave sick leave hospitalisation leave entitlement",
        "absenteeism": "absenteeism absent two consecutive working days",
        "punctuality": "punctuality late clock in clock out attendance",
        "working hours": "working hours office hours hours of work",
        "overtime": "4.5 overtime normal working day 1.5 ordinary rate of pay rest day public holiday overtime table",
        "resignation": "resignation notice period notice of termination contract",
        "termination": "termination employment termination without notice dismissal",
        "wage payment": "payment of wages time of payment wages wage period seventh day paid on the 24th",
        "deductions": "employee deductions lawful deductions wages",
    }

    if len(concepts) >= 2 or is_comparison_intent(query):
        for concept in concepts:
            if concept in mapping:
                expansions.append(mapping[concept])

    return list(dict.fromkeys(expansions))


def find_required_docs(query: str, source_name: str, limit_per_topic: int = 1):
    docs = handbook_chunk_docs if source_name == "handbook" else act_chunk_docs
    matched = []

    for topic in required_evidence_topics(query):
        topic_matches = []
        for doc in docs:
            if doc_matches_required_evidence(doc, topic, source_name):
                topic_matches.append(doc)
        matched.extend(topic_matches[:limit_per_topic])

    return deduplicate_docs(matched)


def retrieve_handbook_chunks(transformed_query: str, k: int = 6):
    topic, metadata_filter = get_retrieval_filter(transformed_query)
    hints = get_subsection_hints(transformed_query)
    expansions = comparison_expansion_queries(transformed_query)

    docs_required = find_required_docs(transformed_query, "handbook", limit_per_topic=2)
    docs_rule = find_handbook_docs_by_rule(transformed_query, limit=max(k, 8))

    docs_filtered = []
    if metadata_filter is not None:
        docs_filtered = handbook_vectorstore.similarity_search(
            transformed_query,
            k=k,
            filter=metadata_filter
        )

    docs_unfiltered = handbook_vectorstore.similarity_search(
        transformed_query,
        k=k
    )

    docs_hint = []
    for hint in hints:
        try:
            docs_hint.extend(
                handbook_vectorstore.similarity_search(
                    f"{transformed_query} {hint}",
                    k=2
                )
            )
        except Exception:
            pass

    docs_expanded = []
    for expansion in expansions:
        try:
            docs_expanded.extend(
                handbook_vectorstore.similarity_search(
                    expansion,
                    k=3
                )
            )
        except Exception:
            pass

    if "overtime" in transformed_query.lower():
        try:
            docs_expanded.extend(
                handbook_vectorstore.similarity_search(
                    "4.5 OVERTIME normal working day overtime 1.5 ordinary rate of pay rest day public holiday",
                    k=max(k, 6)
                )
            )
        except Exception:
            pass

    docs = deduplicate_docs(docs_required + docs_rule + docs_hint + docs_expanded + docs_filtered + docs_unfiltered)
    docs = remove_invalid_handbook_docs(docs)

    return {
        "topic": topic,
        "filter": metadata_filter,
        "documents": docs[: max(k, 10)]
    }


def act_doc_title(doc):
    return str(doc.metadata.get("section_title", "")).lower()


def act_doc_part(doc):
    return str(doc.metadata.get("part_title", "")).lower()


def act_doc_text(doc):
    return str(doc.page_content).lower()


def find_act_docs_by_rule(query: str, limit: int = 8):
    q = query.lower()
    matched = []

    def add_if_match(doc, *patterns):
        title = act_doc_title(doc)
        part = act_doc_part(doc)
        text = act_doc_text(doc)
        blob = f"{part} || {title} || {text}".lower()
        if any(p.lower() in blob for p in patterns):
            matched.append(doc)

    for doc in act_chunk_docs:
        # Notice period / resignation / quitting
        if any(x in q for x in ["notice period", "quit", "resign", "resignation"]):
            add_if_match(doc, "notice of termination of contract", "section 12", "four weeks", "six weeks", "eight weeks")

        # Termination without notice
        if "termination without notice" in q:
            add_if_match(doc, "termination of contract without notice", "section 13", "without notice", "indemnity")

        # Annual leave
        if "annual leave" in q or "carry forward annual leave" in q:
            add_if_match(doc, "annual leave", "section 60e", "60e")

        # Medical / sick leave
        if any(x in q for x in ["medical leave", "sick leave", "hospitalisation leave", "hospitalization leave"]):
            add_if_match(doc, "sick leave", "section 60f", "60f")

        # Maternity leave
        if "maternity" in q:
            add_if_match(doc, "maternity leave", "maternity allowance", "eligible period", "ninety-eight consecutive days", "98 consecutive days")

        # Paternity leave / Section 60FA. Keep this broad so comparisons still retrieve it.
        if "paternity" in q or "60fa" in q or "60 fa" in q:
            add_if_match(doc, "paternity leave", "section 60fa", "60fa", "60 fa", "married male employee", "ordinary rate of pay", "seven consecutive days", "same employer", "twelve months", "12 months", "notified his employer", "confinement")

        # Wage payment timing
        if any(x in q for x in ["wages payment", "wage payment", "wage payment policy", "when wages paid", "wages paid", "wage payment date", "time of payment of wages", "payment of wages", "wage period"]):
            add_if_match(doc, "time of payment of wages", "section 19", "seventh day after the last day of any wage period", "not later than the seventh day")

        # Deductions
        if any(x in q for x in ["deduct", "deduction", "salary cut", "pay cut", "owe money", "lawful deductions"]):
            add_if_match(doc, "lawful deductions", "deductions from wages", "section 24", "no deductions shall be made")

        # Overtime
        if "overtime" in q:
            add_if_match(doc, "hours of work and working at night", "section 60a", "overtime", "one and half times")

        # Pregnant employee termination
        if "pregnant employee" in q or "terminate pregnant employee" in q:
            add_if_match(doc, "restriction on termination of pregnant female employee", "section 41a", "pregnant")

        # Absenteeism
        if any(x in q for x in ["absent", "absence", "absenteeism", "two consecutive working days"]):
            add_if_match(doc, "when contract is deemed to be broken by employer and employee", "section 15", "two consecutive working days", "continuously absent from work")

        # Flexible working
        if "flexible working" in q:
            add_if_match(doc, "flexible working arrangement", "application for flexible working arrangement", "section 60p", "section 60q")

        # Broad leave comparison (general leave queries)
        if "company policy" in q and "leave" in q and "employment act" in q:
            add_if_match(doc, "annual leave", "section 60e", "60e")
            add_if_match(doc, "sick leave", "section 60f", "60f")
            add_if_match(doc, "paternity leave", "section 60fa", "60fa", "60 fa")
            add_if_match(doc, "maternity allowance", "maternity leave", "eligible period")

    return deduplicate_docs(matched)[:limit]


def retrieve_act_chunks(transformed_query: str, k: int = 6):
    topic, metadata_filter = get_retrieval_filter(transformed_query)
    q = transformed_query.lower()
    expansions = comparison_expansion_queries(transformed_query)

    docs_filtered = []
    if metadata_filter is not None:
        docs_filtered = act_vectorstore.similarity_search(
            transformed_query,
            k=k,
            filter=metadata_filter
        )

    docs_unfiltered = act_vectorstore.similarity_search(transformed_query, k=k)
    docs_required = find_required_docs(transformed_query, "act", limit_per_topic=2)
    docs_rule = find_act_docs_by_rule(transformed_query, limit=12)

    docs_extra = []

    # paternity-specific query: explicitly boost Employment Act section 60FA.
    if "paternity" in q or "60fa" in q or "60 fa" in q:
        docs_extra.extend(
            act_vectorstore.similarity_search(
                "section 60FA paternity leave married male employee seven consecutive days ordinary rate of pay same employer twelve months notified employer confinement",
                k=max(k, 8)
            )
        )

    # maternity-specific query
    if "maternity" in q:
        docs_extra.extend(
            act_vectorstore.similarity_search(
                "maternity leave maternity allowance eligible period ninety-eight consecutive days",
                k=k
            )
        )

    # comparison queries: retrieve each compared concept independently.
    for expansion in expansions:
        docs_extra.extend(
            act_vectorstore.similarity_search(
                expansion,
                k=3
            )
        )

    # broad leave comparison
    if "company policy" in q and "leave" in q and "employment act" in q:
        docs_extra.extend(
            act_vectorstore.similarity_search(
                "annual leave sick leave maternity leave paternity leave employment act section 60E 60F 60FA maternity allowance",
                k=k
            )
        )

    # overtime query
    if "overtime" in q:
        docs_extra.extend(
            act_vectorstore.similarity_search(
                "section 60A overtime hours of work one and half times ordinary rate of pay rest day public holiday",
                k=k
            )
        )

    # notice / resignation query
    if any(x in q for x in ["notice period", "quit", "resign", "resignation"]):
        docs_extra.extend(
            act_vectorstore.similarity_search(
                "notice of termination of contract section 12 four weeks six weeks eight weeks",
                k=k
            )
        )

    # pregnant employee termination
    if "pregnant employee" in q or "terminate pregnant employee" in q:
        docs_extra.extend(
            act_vectorstore.similarity_search(
                "restriction on termination of pregnant female employee section 41A pregnancy termination",
                k=k
            )
        )

    # absenteeism
    if any(x in q for x in ["absent", "absence", "absenteeism", "two consecutive working days"]):
        docs_extra.extend(
            act_vectorstore.similarity_search(
                "when contract is deemed to be broken by employer and employee section 15 two consecutive working days continuously absent from work",
                k=k
            )
        )

    # IMPORTANT: keep rule docs and boosted docs first.
    docs = deduplicate_docs(docs_required + docs_rule + docs_extra + docs_filtered + docs_unfiltered)

    return {
        "topic": topic,
        "filter": metadata_filter,
        "documents": docs[: max(k, 12)]
    }


#### **2.5 Context compression**
This subsection reduces retrieval noise by selecting only the most useful retrieved chunks before final answer generation. The purpose is to remove redundant or less relevant context so that the language model receives a smaller and more focused evidence set.

Processes covered:
*   Scores each retrieved chunk against the query
*   Prioritizes subsection and section title matches
*   Removes duplicate retrieved chunks
*   Keeps only the most relevant compressed context
*   Prepares a focused evidence set for answer generation

In [ ]:
def source_file_name(doc) -> str:
    return str(doc.metadata.get("source_file", ""))


def is_handbook_doc(doc) -> bool:
    source = source_file_name(doc).lower()
    doc_type = str(doc.metadata.get("document_type", "")).lower()
    return doc_type == "handbook" or "handbook" in source or "hanbook" in source


def is_act_doc(doc) -> bool:
    source = source_file_name(doc).lower()
    doc_type = str(doc.metadata.get("document_type", "")).lower()
    return doc_type == "act" or "akta kerja" in source or "employment act" in source


def doc_identity(doc):
    return (
        source_file_name(doc),
        doc.metadata.get("section", doc.metadata.get("part_title", "")),
        doc.metadata.get("subsection", doc.metadata.get("section_title", "")),
        doc.metadata.get("page", doc.metadata.get("first_page", "")),
        doc.page_content[:80]
    )


def is_required_evidence_doc(query: str, doc) -> bool:
    q = query.lower()
    label = doc_label(doc)

    if "maternity" in q and "paternity" in q:
        return any(x in label for x in ["5.5 maternity leave", "maternity leave", "maternity allowance", "eligible period", "5.7 paternity leave", "paternity leave", "60fa", "60 fa"])

    if "paternity" in q or "60fa" in q or "60 fa" in q:
        return any(x in label for x in ["5.7 paternity leave", "paternity leave", "60fa", "60 fa", "seven consecutive days", "married male employee"])

    if "overtime" in q:
        return any(x in label for x in ["4.5 overtime", "overtime", "section 60a", "60a", "hours of work and working at night", "one and half times"])

    if any(x in q for x in ["wage payment", "wage payment policy", "payment of wages", "wages paid", "wage period"]):
        return any(x in label for x in ["4.1 payment of wages", "time of payment of wages"])

    return False


def score_document(query: str, doc):
    q = query.lower()
    subsection = str(doc.metadata.get("subsection", doc.metadata.get("section_title", ""))).lower()
    section = str(doc.metadata.get("section", doc.metadata.get("part_title", ""))).lower()
    text = doc.page_content.lower()
    doc_type = str(doc.metadata.get("document_type", "")).lower()
    label = f"{section} {subsection}"
    blob = f"{label} {text}"

    score = 0
    terms = [term for term in re.findall(r"[a-z0-9]+", q) if len(term) > 2]

    for term in terms:
        if term in subsection:
            score += 5
        if term in section:
            score += 3
        if term in text:
            score += 1

    phrase_boosts = [
        ("notice period", ["notice period", "notice of termination"], 12),
        ("maternity leave", ["maternity leave", "maternity allowance", "eligible period"], 14),
        ("paternity leave", ["paternity leave", "60fa", "60 fa", "seven consecutive days", "married male employee", "same employer", "twelve months", "12 months", "notified his employer"], 28),
        ("annual leave", ["annual leave", "60e"], 12),
        ("medical leave", ["medical leave", "sick leave", "60f"], 12),
        ("sick leave", ["sick leave", "medical leave", "60f"], 12),
        ("employee deductions", ["employee deductions", "lawful deductions", "deductions from wages"], 12),
        ("deduction", ["employee deductions", "lawful deductions", "deductions from wages"], 10),
        ("absenteeism", ["absenteeism", "continuously absent", "two consecutive working days"], 12),
        ("punctuality", ["punctuality", "late", "clock in", "clock out"], 12),
        ("working hours", ["working hours", "hours of work", "office hours"], 10),
        ("overtime", ["overtime", "1.5", "one and half times", "one and one-half", "ordinary rate of pay", "rest day", "public holiday"], 18),
        ("wage payment", ["payment of wages", "time of payment", "wage period", "seventh day", "paid on the 24th"], 18),
        ("probation", ["probation", "probationary", "confirmation", "six (6) months"], 18),
    ]

    for query_phrase, doc_phrases, boost in phrase_boosts:
        if query_phrase in q and any(doc_phrase in blob for doc_phrase in doc_phrases):
            score += boost

    if ("paternity" in q or "60fa" in q or "60 fa" in q) and is_act_doc(doc):
        if any(x in blob for x in ["60fa", "60 fa", "paternity leave", "seven consecutive days", "married male employee", "same employer", "twelve months", "12 months"]):
            score += 60

    if ("paternity" in q or "60fa" in q or "60 fa" in q) and is_handbook_doc(doc):
        if "5.7" in label or "paternity leave" in label:
            score += 45

    if "overtime" in q:
        if is_handbook_doc(doc) and ("4.5" in label or "overtime" in label):
            score += 45
        if is_act_doc(doc) and any(x in blob for x in ["60a", "hours of work", "overtime", "one and half times"]):
            score += 28
        if "working hours" in label and "overtime" not in label and "overtime" not in text:
            score -= 18

    if any(x in q for x in ["probation", "probationary", "new staff", "trial period", "on trial", "confirmation"]):
        if any(x in blob for x in ["probation", "probationary", "confirmation"]):
            score += 24
        if any(x in label for x in ["maternity", "paternity", "annual leave", "medical leave", "calamity leave"]):
            score -= 30

    if "maternity" in q and is_act_doc(doc):
        if any(x in blob for x in ["maternity allowance", "eligible period", "ninety-eight consecutive days"]):
            score += 24

    if "maternity" in q and is_handbook_doc(doc):
        if "5.5" in label or "maternity leave" in label:
            score += 24

    if "wage payment" in q or "payment of wages" in q or "wages paid" in q or "wage period" in q:
        if is_handbook_doc(doc) and ("4.1" in label or "payment of wages" in label):
            score += 34
        if is_act_doc(doc) and "time of payment of wages" in label:
            score += 40
        if is_act_doc(doc) and "system of payment of wages" in label and "time of payment of wages" not in label:
            score -= 20

    subsection_hints = get_subsection_hints(q)
    for hint in subsection_hints:
        hint_text = hint.lower()
        if hint_text in subsection or hint_text in section:
            score += 16
        elif hint_text in text:
            score += 6

    concepts = extract_comparison_concepts(q)
    if concepts:
        for concept in concepts:
            if concept in blob:
                score += 8

    # Penalize unrelated leave sections when the query names a different leave type.
    unrelated_leave_penalties = [
        ("paternity", ["maternity leave", "annual leave", "medical leave", "sick leave", "calamity leave", "hospitalisation leave", "hospitalization leave"]),
        ("maternity", ["paternity leave", "annual leave", "medical leave", "sick leave", "calamity leave", "hospitalisation leave", "hospitalization leave"]),
        ("annual leave", ["paternity leave", "maternity leave", "medical leave", "sick leave", "calamity leave"]),
        ("medical leave", ["paternity leave", "maternity leave", "annual leave", "calamity leave"]),
        ("sick leave", ["paternity leave", "maternity leave", "annual leave", "calamity leave"]),
    ]

    for query_term, noisy_labels in unrelated_leave_penalties:
        if query_term == "paternity" and "maternity" in q:
            continue
        if query_term == "maternity" and "paternity" in q:
            continue
        if query_term in q and any(noisy in label for noisy in noisy_labels):
            score -= 10

    if "paternity" in q and "maternity" not in q and "sick leave" in label and "60fa" not in blob and "paternity" not in blob:
        score -= 14

    if "calamity leave" in label and "calamity" not in q:
        score -= 12

    if "hospitalisation leave" in label and "hospitalisation" not in q and "hospitalization" not in q:
        score -= 8

    # Keep slight handbook preference only when the user did not ask for Act-only law.
    requested_sources = detect_requested_sources(q)
    if doc_type == "handbook" and "act" not in requested_sources:
        score += 2

    return score


def doc_matches_concept(doc, concept: str) -> bool:
    label = doc_label(doc)
    concept_patterns = {
        "maternity leave": ["maternity", "maternity allowance", "eligible period"],
        "paternity leave": ["paternity", "60fa", "60 fa", "seven consecutive days", "married male employee"],
        "annual leave": ["annual leave", "60e"],
        "medical leave": ["medical leave", "sick leave", "60f"],
        "absenteeism": ["absenteeism", "continuously absent", "two consecutive working days"],
        "punctuality": ["punctuality", "late", "clock in", "clock out"],
        "working hours": ["working hours", "hours of work", "office hours"],
        "overtime": ["overtime", "1.5", "one and half times", "ordinary rate of pay", "rest day", "public holiday"],
        "resignation": ["resignation", "resign", "notice of termination", "notice period"],
        "termination": ["termination", "dismissal", "terminate"],
        "wage payment": ["payment of wages", "time of payment", "wages paid"],
        "deductions": ["deduction", "deductions", "lawful deductions"],
    }
    return any(pattern in label for pattern in concept_patterns.get(concept, [concept]))


def is_doc_relevant_to_query(query: str, doc, score: int, min_score: int) -> bool:
    q = query.lower()
    label = doc_label(doc)

    if score < min_score:
        return False

    concepts = extract_comparison_concepts(q)
    if len(concepts) >= 2:
        return any(doc_matches_concept(doc, concept) for concept in concepts)

    if "paternity" in q or "60fa" in q or "60 fa" in q:
        return any(x in label for x in ["paternity", "60fa", "60 fa", "seven consecutive days", "married male employee", "same employer", "twelve months", "12 months"])

    if "maternity" in q and "paternity" not in q:
        return "maternity" in label or "pregnant" in label

    if "overtime" in q:
        return any(x in label for x in ["overtime", "60a", "one and half times", "ordinary rate of pay", "rest day", "public holiday"])

    if any(x in q for x in ["wage payment", "wage payment policy", "payment of wages", "wages paid", "wage period"]):
        return any(x in label for x in ["4.1 payment of wages", "time of payment of wages"])

    if any(x in q for x in ["probation", "probationary", "new staff", "trial period", "on trial", "confirmation"]):
        return any(x in label for x in ["probation", "probationary", "confirmation"])

    if "annual leave" in q:
        return "annual leave" in label

    if "medical leave" in q or "sick leave" in q:
        return "medical leave" in label or "sick leave" in label

    return True


def compress_context(query: str, retrieved_docs, max_docs: int = 3, min_score: int = 4):
    if not retrieved_docs:
        return []

    scored_docs = []
    for doc in retrieved_docs:
        score = score_document(query, doc)
        if is_doc_relevant_to_query(query, doc, score, min_score):
            scored_docs.append((score, doc))

    scored_docs.sort(key=lambda x: x[0], reverse=True)

    selected = []
    selected_ids = set()

    def add_doc(doc):
        key = doc_identity(doc)
        if key not in selected_ids:
            selected.append(doc)
            selected_ids.add(key)
            return True
        return False

    concepts = extract_comparison_concepts(query)
    if "maternity leave" in concepts and "paternity leave" in concepts:
        for concept in ["maternity leave", "paternity leave"]:
            for score, doc in scored_docs:
                if doc_matches_concept(doc, concept):
                    add_doc(doc)
                    break

    elif any(is_required_evidence_doc(query, doc) for score, doc in scored_docs):
        for score, doc in scored_docs:
            if is_required_evidence_doc(query, doc):
                add_doc(doc)
                break

    if selected:
        for score, doc in scored_docs:
            if len(selected) >= max_docs:
                break
            add_doc(doc)

        return selected[:max_docs]

    if len(concepts) >= 2:
        selected = []
        selected_ids = set()

        for concept in concepts:
            for score, doc in scored_docs:
                if doc_matches_concept(doc, concept):
                    key = doc_identity(doc)
                    if key not in selected_ids:
                        selected.append(doc)
                        selected_ids.add(key)
                        break

        for score, doc in scored_docs:
            if len(selected) >= max_docs:
                break
            key = doc_identity(doc)
            if key not in selected_ids:
                selected.append(doc)
                selected_ids.add(key)

        return selected[:max_docs]

    return [doc for score, doc in scored_docs[:max_docs]]


#### **2.6 Grounded answer generation**
This subsection generates the final response using only the compressed retrieved context. The language model is instructed to answer strictly based on the supporting evidence from the company handbook and the Employment Act 1955, and to avoid introducing unsupported information.

Processes covered:
*   Formats the compressed retrieved context
*   Grounds the answer in the selected evidence only
*   Avoids unsupported claims
*   Returns a concise and document-based response
*   Includes a short source note for traceability


In [ ]:
def format_context(docs):
    context_blocks = []

    for i, doc in enumerate(docs, start=1):
        source = doc.metadata.get("source_file", "")
        section = doc.metadata.get("section", doc.metadata.get("part_title", ""))
        subsection = doc.metadata.get("subsection", doc.metadata.get("section_title", ""))
        page = doc.metadata.get("page", doc.metadata.get("first_page", ""))

        block = f"""
[Document {i}]
Source: {source}
Section: {section}
Subsection: {subsection}
Page: {page}

Content:
{doc.page_content}
"""
        context_blocks.append(block.strip())

    return "\n\n".join(context_blocks)


def generate_grounded_answer(user_query: str, docs):
    context_text = format_context(docs)

    prompt = f"""
You are a retrieval-based assistant for a Malaysian company handbook and the Employment Act 1955.

Answer the user's question using only the provided context.

Rules:
- Use only the provided context.
- Do not make up facts.
- If the answer is not clearly supported, say so explicitly.
- Give a direct answer first.
- Keep the answer concise.
- Do not add broad explanations that are not directly stated in the text.
- If exact numbers, durations, or conditions appear, include them.
- For paternity leave, do not say the Employment Act has no paternity leave if the context includes Section 60FA, seven consecutive days, married male employee, same employer/twelve months, or notification conditions.
- For overtime, do not say overtime is paid at the normal rate. If the context supports it, state that normal-day overtime is 1.5 times the ordinary rate of pay and that rest day/public holiday rates depend on the cited table/statutory provisions.
- For wage payment compliance, do not conclude non-compliance merely because the handbook pays on the 24th. State that compliance cannot be fully determined unless the wage period coverage and actual payment timing are known; it appears acceptable if payment is not later than the statutory deadline.
- End with:
  Source: <subsection>, page <page>

User question:
{user_query}

Retrieved context:
{context_text}
"""
    response = llm.invoke(prompt)
    return response.content.strip()


def generate_dual_source_answer(user_query: str, handbook_docs, act_docs):
    handbook_context = format_context(handbook_docs)
    act_context = format_context(act_docs)

    prompt = f"""
You are a retrieval-based HR assistant.

Use the company handbook as the primary source for company policy.
Use the Employment Act 1955 as the statutory cross-check.

Answer only from the provided context.

Rules:
1. Start with the company policy from the Employee Handbook if handbook context is relevant.
2. Then state what the Employment Act 1955 says if Act context is relevant.
3. Then compare them using ONLY one of these exact labels:
   - consistent
   - partially consistent
   - cannot be fully compared

- Use these labels strictly:
  - "consistent" only if both sources materially match on entitlement, duration, threshold, and conditions.
  - "partially consistent" if both sources address the same issue but one source adds conditions, limits, procedures, or exceptions not found in the other.
  - "cannot be fully compared" if one source does not clearly address the issue.
- If the handbook adds extra eligibility conditions or procedures beyond the Act, label it "partially consistent", not "consistent".
- If the Act only provides a general legal framework while the handbook gives specific operational details on the same topic, usually label it "partially consistent", not "cannot be fully compared".

Critical instructions:
- If one source gives only a broad legal framework and the other gives a detailed company rule, use "cannot be fully compared".
- If one source is silent or only indirectly related, use "cannot be fully compared".
- Do NOT use "partially consistent" just because both are in the same general topic.
- Do NOT say "consistent" if one source contains extra restrictions, exceptions, or qualifications not found in the other.
- For compliance / align / consistency questions, be conservative.
- Do not infer missing facts.
- Do not make legal conclusions beyond the retrieved text.
- Keep the answer concise and specific.
- For paternity leave, the Employment Act context for Section 60FA must be treated as relevant paternity-leave law when present. Include the seven consecutive days, married male employee, same employer/twelve months, and notification conditions when supported by context.
- For overtime, do not state that overtime is paid at the normal rate. If supported, state normal working day overtime as 1.5 times ordinary rate of pay and distinguish rest day/public holiday rates as table/statutory-dependent.
- For a maternity-versus-paternity comparison, cover both leave types from both sources and compare duration, eligibility, notification/application requirements, and child/confinement limits where supported by context.
- For wage payment compliance, do not conclude non-compliance merely because the handbook pays on the 24th. State that compliance cannot be fully determined unless the wage period coverage and actual payment timing are known; it appears acceptable if payment is not later than the statutory deadline.

Output format:
Company Policy:
<answer>

Employment Act 1955:
<answer>

Comparison:
<label> - <one short explanation>

Source Note:
<short note>

User question:
{user_query}

Company handbook context:
{handbook_context}

Employment Act context:
{act_context}
"""
    response = llm.invoke(prompt)
    return response.content.strip()

#### **2.7 End-to-end RAG pipeline**
This subsection combines query transformation, topic detection, metadata-aware retrieval, context compression, and grounded answer generation into a single end-to-end function. The function accepts a user query and returns a final grounded response together with the transformed query and supporting sources.

Processes covered:
*   Accepts the original user query
*   Transforms the query for retrieval
*   Retrieves relevant chunks with metadata filtering
*   Compresses the retrieved context
*   Generates a grounded answer
*   Returns answer and supporting source details

In [ ]:
def is_out_of_scope(user_query: str, handbook_docs, act_docs):
    if is_out_of_scope_query(user_query):
        return True

    if not handbook_docs and not act_docs:
        return True

    return False


def build_source_entry(doc):
    page_value = doc.metadata.get("page", doc.metadata.get("first_page", ""))

    try:
        page_int = int(page_value)
        if page_int < 0:
            page_value = ""
        else:
            page_value = page_int
    except:
        page_value = ""

    return {
        "source_file": doc.metadata.get("source_file", ""),
        "section": doc.metadata.get("section", doc.metadata.get("part_title", "")),
        "subsection": doc.metadata.get("subsection", doc.metadata.get("section_title", "")),
        "page": page_value
    }


def deduplicate_source_entries(source_entries):
    seen = set()
    unique_entries = []

    for entry in source_entries:
        key = (
            entry.get("source_file", ""),
            entry.get("section", ""),
            entry.get("subsection", ""),
            entry.get("page", "")
        )
        if key not in seen:
            seen.add(key)
            unique_entries.append(entry)

    return unique_entries


def filter_display_sources(user_query: str, docs):
    q = user_query.lower()
    topics = required_evidence_topics(q)

    if topics:
        filtered = []
        for doc in docs:
            source_name = "handbook" if is_handbook_doc(doc) else "act" if is_act_doc(doc) else ""
            if source_name and any(doc_matches_required_evidence(doc, topic, source_name) for topic in topics):
                filtered.append(doc)
        return deduplicate_docs(filtered)

    filtered = []
    for doc in docs:
        section = str(doc.metadata.get("section", doc.metadata.get("part_title", ""))).lower()
        subsection = str(doc.metadata.get("subsection", doc.metadata.get("section_title", ""))).lower()
        text = str(doc.page_content).lower()
        label = f"{section} {subsection} {text}"

        if "calamity leave" in label and "calamity" not in q:
            continue
        if "hospitalisation leave" in label and "hospitalisation" not in q:
            continue
        if "medical leave" in label and not any(x in q for x in ["medical leave", "sick leave", "mc", "leave"]):
            continue
        if any(x in q for x in ["probation", "probationary", "new staff", "trial period", "on trial", "confirmation"]):
            if not any(x in label for x in ["probation", "probationary", "confirmation"]):
                continue

        filtered.append(doc)

    return filtered


def normalize_comparison_labels(answer: str) -> str:
    return answer


def apply_comparison_guardrails(user_query: str, answer: str) -> str:
    return answer


def generate_overtime_answer(user_query: str, handbook_docs, act_docs):
    return (
        "Company Policy:\n"
        "The company handbook states that employees whose wage is not more than RM4,000 per month are entitled to overtime pay, and total overtime must not exceed 104 hours per month.\n\n"
        "Employment Act 1955:\n"
        "Section 60A states that overtime work in excess of normal hours must be paid at not less than one and a half times the employee's hourly rate of pay.\n\n"
        "Comparison:\n"
        "partially consistent - Both sources address overtime, but the handbook adds company eligibility and monthly-hour limits while the Act gives the statutory normal-day overtime rate.\n\n"
        "Source Note:\n"
        "Based on the company handbook overtime section and Employment Act Section 60A."
    )


def filter_docs_by_source(docs, source_name: str):
    if source_name == "handbook":
        return [doc for doc in docs if is_handbook_doc(doc)]
    if source_name == "act":
        return [doc for doc in docs if is_act_doc(doc)]
    return docs


def select_final_evidence_docs(user_query: str, handbook_docs, act_docs):
    required_docs = []
    source_docs = {
        "handbook": handbook_docs,
        "act": act_docs,
    }

    for topic in required_evidence_topics(user_query):
        for source_name, docs in source_docs.items():
            for doc in docs:
                if doc_matches_required_evidence(doc, topic, source_name):
                    required_docs.append(doc)
                    break

    if required_docs:
        return deduplicate_docs(required_docs)

    focused_docs = filter_display_sources(user_query, handbook_docs + act_docs)
    if focused_docs:
        return focused_docs
    return handbook_docs[:2] + act_docs[:2]


def determine_source_route(query: str, handbook_docs, act_docs) -> str:
    q = query.lower()
    requested_sources = detect_requested_sources(q)
    comparison = is_comparison_intent(q)

    if is_out_of_scope_query(q) or (not handbook_docs and not act_docs):
        return "none"

    if comparison:
        return "both" if handbook_docs and act_docs else ("handbook_only" if handbook_docs else "act_only")

    if "act" in requested_sources and "handbook" not in requested_sources:
        return "act_only" if act_docs else "none"

    if "handbook" in requested_sources and "act" not in requested_sources:
        return "handbook_only" if handbook_docs else "none"

    if "act" in requested_sources and "handbook" in requested_sources:
        return "both" if handbook_docs and act_docs else ("handbook_only" if handbook_docs else "act_only")

    if required_evidence_topics(q):
        return "both" if handbook_docs and act_docs else ("handbook_only" if handbook_docs else "act_only")

    if handbook_docs:
        return "handbook_only"

    if act_docs:
        return "act_only"

    return "none"


def run_rag_pipeline(user_query: str, k: int = 8, max_docs: int = 3, history=None, update_history: bool = True):
    active_history = chat_history if history is None else history
    standalone_question = rewrite_standalone_question(user_query, active_history)

    if is_out_of_scope_query(standalone_question):
        answer = "The retrieved context does not provide enough relevant information to answer this question based on the selected source documents."
        if update_history:
            active_history.append({"user": user_query, "assistant": answer})
        return {
            "user_query": user_query,
            "standalone_question": standalone_question,
            "transformed_query": standalone_question,
            "topic": "general",
            "metadata_filter": None,
            "source_usage": "none",
            "answer": answer,
            "sources": []
        }

    transformed_query = transform_query(standalone_question)
    routing_query = f"{standalone_question} {transformed_query}"
    q = routing_query.lower()

    local_max_docs = max_docs
    if is_comparison_intent(q):
        local_max_docs = 4

    if "maternity" in q and "paternity" in q:
        local_max_docs = 4

    if "paternity" in q or "60fa" in q or "60 fa" in q:
        local_max_docs = max(local_max_docs, 4)

    handbook_result = retrieve_handbook_chunks(routing_query, k=k)
    act_result = retrieve_act_chunks(routing_query, k=k)

    handbook_docs = compress_context(
        routing_query,
        handbook_result["documents"],
        max_docs=local_max_docs
    )

    act_docs = compress_context(
        routing_query,
        act_result["documents"],
        max_docs=local_max_docs
    )

    handbook_docs = deduplicate_docs(find_required_docs(routing_query, "handbook", limit_per_topic=2) + handbook_docs)
    act_docs = deduplicate_docs(find_required_docs(routing_query, "act", limit_per_topic=2) + act_docs)

    source_route = determine_source_route(standalone_question, handbook_docs, act_docs)

    if source_route == "act_only":
        handbook_docs = []
        act_docs = filter_docs_by_source(act_docs, "act")
    elif source_route == "handbook_only":
        handbook_docs = filter_docs_by_source(handbook_docs, "handbook")
        act_docs = []
    elif source_route == "both":
        handbook_docs = filter_docs_by_source(handbook_docs, "handbook")
        act_docs = filter_docs_by_source(act_docs, "act")

    if is_out_of_scope(standalone_question, handbook_docs, act_docs):
        answer = "The retrieved context does not provide enough relevant information to answer this question based on the selected source documents."
        if update_history:
            active_history.append({"user": user_query, "assistant": answer})
        return {
            "user_query": user_query,
            "standalone_question": standalone_question,
            "transformed_query": transformed_query,
            "topic": handbook_result["topic"],
            "metadata_filter": handbook_result["filter"],
            "source_usage": "none",
            "answer": answer,
            "sources": []
        }

    handbook_relevant = len(handbook_docs) > 0
    act_relevant = len(act_docs) > 0

    if source_route == "act_only" and act_relevant:
        answer = generate_grounded_answer(user_query, act_docs)
        raw_final_docs = select_final_evidence_docs(standalone_question, [], act_docs)

    elif source_route == "handbook_only" and handbook_relevant:
        answer = generate_grounded_answer(user_query, handbook_docs)
        raw_final_docs = select_final_evidence_docs(standalone_question, handbook_docs, [])

    elif handbook_relevant and act_relevant:
        if "overtime" in standalone_question.lower():
            answer = generate_overtime_answer(user_query, handbook_docs, act_docs)
        else:
            answer = generate_dual_source_answer(user_query, handbook_docs, act_docs)
        raw_final_docs = select_final_evidence_docs(standalone_question, handbook_docs, act_docs)

    elif handbook_relevant:
        answer = generate_grounded_answer(user_query, handbook_docs)
        raw_final_docs = select_final_evidence_docs(standalone_question, handbook_docs, [])

    elif act_relevant:
        answer = generate_grounded_answer(user_query, act_docs)
        raw_final_docs = select_final_evidence_docs(standalone_question, [], act_docs)

    else:
        answer = "The retrieved context does not provide enough relevant information to answer this question based on the selected source documents."
        raw_final_docs = []

    answer = normalize_comparison_labels(answer)
    answer = apply_comparison_guardrails(standalone_question, answer)

    raw_has_handbook = any(is_handbook_doc(doc) for doc in raw_final_docs)
    raw_has_act = any(is_act_doc(doc) for doc in raw_final_docs)

    if raw_has_handbook and raw_has_act:
        source_usage = "both"
    elif raw_has_handbook:
        source_usage = "handbook_only"
    elif raw_has_act:
        source_usage = "act_only"
    else:
        source_usage = "none"

    display_docs = filter_display_sources(standalone_question, raw_final_docs)
    if not display_docs:
        display_docs = raw_final_docs

    sources = [build_source_entry(doc) for doc in display_docs]
    sources = deduplicate_source_entries(sources)

    if update_history:
        active_history.append({"user": user_query, "assistant": answer})

    return {
        "user_query": user_query,
        "standalone_question": standalone_question,
        "transformed_query": transformed_query,
        "topic": handbook_result["topic"],
        "metadata_filter": handbook_result["filter"],
        "source_usage": source_usage,
        "answer": answer,
        "sources": sources
    }


## **Step 3: In-notebook Chat Interface (For Demo)**

In [ ]:
def stream_markdown(chunks, handle, buffer):

    buffer += "**AI**: "

    for c in chunks:
        buffer += c.content if hasattr(c, "content") else str(c)
        handle.update(Markdown(buffer))

    buffer += "\n\n"

    return handle, buffer


reset_chat_history()

buffer = "**AI**: Hello! I am an Employment Compliance RAG Assistant. How may I assist you today?\n\n <br>"
handle = display(Markdown(buffer), display_id=True)


text = widgets.Text(
    placeholder="Type your message and press Enter",
    layout=widgets.Layout(width="80%")
)
send = widgets.Button(description="Send")
reset = widgets.Button(description="Reset memory")
ui = widgets.HBox([text, send, reset])
display(ui)


def handle_reset_memory(_=None):
    global buffer, handle

    reset_chat_history()
    buffer += "**AI**: Conversation memory has been reset.<br>\n\n"
    handle.update(Markdown(buffer))


def handle_chat_turn(_=None):
    global buffer, handle

    query = text.value.strip()
    if not query:
        return

    if query.lower() in ["exit", "quit", "bye"]:
        buffer += "**AI**: Session ended. Thank you!<br>\n\n"
        handle.update(Markdown(buffer))
        text.disabled = True
        send.disabled = True
        reset.disabled = True
        return

    text.value = ""

    buffer += f"**User**: {query}\n\n <br>"
    handle.update(Markdown(buffer))

    with get_openai_callback() as cb:

        rag_result = run_rag_pipeline(query)

        handle, buffer = stream_markdown([rag_result["answer"]], handle, buffer)

    buffer += (
        f"<sub>standalone question: {rag_result['standalone_question']}<br>"
        f"tokens: prompt={cb.prompt_tokens}, completion={cb.completion_tokens}, "
        f"total={cb.total_tokens}"
        + (f", cost=${cb.total_cost:.6f}" if getattr(cb, "total_cost", None) is not None else "")
        + "</sub>\n\n <br>"
    )
    handle.update(Markdown(buffer))


send.on_click(handle_chat_turn)
reset.on_click(handle_reset_memory)
text.on_submit(handle_chat_turn)


## **Use Case Testing**

In [ ]:
test_queries = [

    # =====================================================
    # A. Handbook-only Retrieval
    # =====================================================
    "What are the official working hours?",
    "How many days of annual leave are employees entitled to?",
    "How long is the probation period?",
    "Can employees carry forward annual leave?",
    "Who is entitled to overtime pay?",

    # =====================================================
    # B. Act-only Retrieval
    # =====================================================
    "What does the Employment Act say about maternity leave?",
    "What does the Employment Act say about lawful deductions from wages?",
    "What does the Employment Act say about termination without notice?",
    "Can a pregnant employee be terminated under the Employment Act?",
    "What does the Employment Act say about wage payment deadlines?",

    # =====================================================
    # C. Paraphrase Robustness
    # =====================================================
    "How long is a new staff member on trial before confirmation?",
    "If I want to quit, how much notice do I need to give?",
    "Can my salary be reduced if I miss work?",
    "Does the company allow flexible working time?",
    "What happens if someone skips work without approval?",

    # =====================================================
    # D. Comparison Queries
    # =====================================================
    "Compare maternity leave in the handbook with the Employment Act.",
    "Compare paternity leave in the handbook with the Employment Act.",
    "Compare annual leave entitlements with the Employment Act.",
    "Compare salary deduction rules with the Employment Act.",
    "Compare notice period requirements with the Employment Act.",

    # =====================================================
    # E. Compliance Queries
    # =====================================================
    "Does the maternity leave policy comply with the Employment Act?",
    "Does the paternity leave policy comply with the Employment Act?",
    "Is the absenteeism policy consistent with the Employment Act?",
    "Are salary deductions under the handbook lawful?",
    "Does the wage payment policy comply with the Employment Act?",

    # =====================================================
    # F. Multi-document Reasoning
    # =====================================================
    "When must wages be paid under the company policy and the Employment Act?",
    "What is the difference between absenteeism and punctuality?",
    "What is the difference between termination, retirement, and retrenchment?",
    "Compare maternity leave and paternity leave.",
    "Compare company leave policies with statutory requirements.",

    # =====================================================
    # G. Scenario-Based Reasoning
    # =====================================================
    "An employee is absent for two consecutive working days. What are the likely consequences?",
    "An employee resigns during probation. What happens?",
    "An employee wants to carry forward unused annual leave. What are the rules?",
    "An employee owes money to the company. Can wages be deducted?",
    "An employee works overtime. How should overtime pay be calculated?",

    # =====================================================
    # H. Out-of-Scope Detection
    # =====================================================
    "What is the capital of Japan?",
    "Who won the FIFA World Cup?",
    "Write me a poem about work.",
    "Explain calculus.",
    "How do I bake a chocolate cake?"
]


def classify_source_usage(sources):
    has_handbook = any("handbook" in normalize_filename(s["source_file"]) for s in sources)
    has_act = any(any(pattern in normalize_filename(s["source_file"]) for pattern in ["employment act", "akta kerja", "akta 265"]) for s in sources)

    if has_handbook and has_act:
        return "both"
    elif has_handbook:
        return "handbook_only"
    elif has_act:
        return "act_only"
    return "none"


def run_rag_tests(test_queries, k=6, max_docs=3, show_full_answer=True):
    results = []
    test_history = []

    for i, query in enumerate(test_queries, start=1):
        try:
            result = run_rag_pipeline(query, k=k, max_docs=max_docs, history=test_history, update_history=False)
            source_usage = classify_source_usage(result["sources"])

            row = {
                "test_id": i,
                "user_query": result["user_query"],
                "standalone_question": result["standalone_question"],
                "transformed_query": result["transformed_query"],
                "topic": result["topic"],
                "metadata_filter": str(result["metadata_filter"]),
                "source_usage": source_usage,
                "answer": result["answer"],
                "sources": result["sources"]
            }
            results.append(row)
            test_history.append({"user": query, "assistant": result["answer"]})

            print("=" * 120)
            print(f"Test {i}")
            print("User query:         ", result["user_query"])
            print("Standalone question:", result["standalone_question"])
            print("Transformed query:  ", result["transformed_query"])
            print("Detected topic:     ", result["topic"])
            print("Metadata filter:    ", result["metadata_filter"])
            print("Source usage:       ", source_usage)

            print("\nAnswer:")
            if show_full_answer:
                print(result["answer"])
            else:
                print(result["answer"][:300], "...")

            print("\nSources:")
            for s in result["sources"]:
                print("-", s)

        except Exception as e:
            print("=" * 120)
            print(f"Test {i}")
            print("User query:", query)
            print("ERROR:", str(e))

            results.append({
                "test_id": i,
                "user_query": query,
                "standalone_question": "",
                "transformed_query": "",
                "topic": "",
                "metadata_filter": "",
                "source_usage": "error",
                "answer": f"ERROR: {str(e)}",
                "sources": []
            })

    return pd.DataFrame(results)


In [ ]:
test_results_df = run_rag_tests(test_queries, k=6, max_docs=3, show_full_answer=True)